In [ ]:
# @title Imports and Notebook Utilities
# Growing NCA-style training with PyTorch
# Differences from original Growing NCA:
# - Circular padding (toroidal) instead of zero-padding
# - No alpha channel / living mask
# - Uses NoiseNCA's 4-filter perception (identity, sobel_x, sobel_y, laplacian)
#
# Same as Growing NCA:
# - Fixed seed initialization (zeros with center activation)
# - Damage augmentation during training
# - Loss-ranked pool sampling
# - Pixel-wise MSE loss
# - Stochastic cell updates (fire_rate = 0.5)
# - Gradient normalization

import os
import io
import PIL.Image, PIL.ImageDraw
import base64
import zipfile
import json
import requests
import numpy as np
import matplotlib.pylab as pl
import glob

from IPython.display import Image, HTML, Markdown, clear_output
from tqdm.notebook import tqdm

import warnings
warnings.filterwarnings("ignore")

os.environ['FFMPEG_BINARY'] = 'ffmpeg'
import moviepy.editor as mvp
from moviepy.video.io.ffmpeg_writer import FFMPEG_VideoWriter


def imread(url, max_size=None, mode=None):
    if isinstance(url, str) and url.startswith(('http:', 'https:')):
        headers = {
            "User-Agent": "Requests in Colab/0.0 (https://colab.research.google.com/; no-reply@google.com) requests/0.0"
        }
        r = requests.get(url, headers=headers)
        f = io.BytesIO(r.content)
    else:
        f = url
    img = PIL.Image.open(f)
    if max_size is not None:
        img.thumbnail((max_size, max_size), PIL.Image.LANCZOS)
    if mode is not None:
        img = img.convert(mode)
    img = np.float32(img) / 255.0
    return img


def np2pil(a):
    if a.dtype in [np.float32, np.float64]:
        a = np.uint8(np.clip(a, 0, 1) * 255)
    return PIL.Image.fromarray(a)


def imwrite(f, a, fmt=None):
    a = np.asarray(a)
    if isinstance(f, str):
        fmt = f.rsplit('.', 1)[-1].lower()
        if fmt == 'jpg':
            fmt = 'jpeg'
        f = open(f, 'wb')
    np2pil(a).save(f, fmt, quality=95)


def imencode(a, fmt='jpeg'):
    a = np.asarray(a)
    if len(a.shape) == 3 and a.shape[-1] == 4:
        fmt = 'png'
    f = io.BytesIO()
    imwrite(f, a, fmt)
    return f.getvalue()


def im2url(a, fmt='jpeg'):
    encoded = imencode(a, fmt)
    base64_byte_string = base64.b64encode(encoded).decode('ascii')
    return 'data:image/' + fmt.upper() + ';base64,' + base64_byte_string


def imshow(a, fmt='jpeg', id=None):
    return display(Image(data=imencode(a, fmt)), display_id=id)


def grab_plot(close=True):
    """Return the current Matplotlib figure as an image"""
    fig = pl.gcf()
    fig.canvas.draw()
    img = np.array(fig.canvas.renderer._renderer)
    a = np.float32(img[..., 3:] / 255.0)
    img = np.uint8(255 * (1.0 - a) + img[..., :3] * a)
    if close:
        pl.close()
    return img


def zoom(img, scale=4):
    img = np.repeat(img, scale, 0)
    img = np.repeat(img, scale, 1)
    return img


class VideoWriter:
    def __init__(self, filename='_autoplay.mp4', fps=30.0, **kw):
        self.writer = None
        self.params = dict(filename=filename, fps=fps, **kw)

    def add(self, img):
        img = np.asarray(img)
        if self.writer is None:
            h, w = img.shape[:2]
            self.writer = FFMPEG_VideoWriter(size=(w, h), **self.params)
        if img.dtype in [np.float32, np.float64]:
            img = np.uint8(img.clip(0, 1) * 255)
        if len(img.shape) == 2:
            img = np.repeat(img[..., None], 3, -1)
        self.writer.write_frame(img)

    def close(self):
        if self.writer:
            self.writer.close()

    def __enter__(self):
        return self

    def __exit__(self, *kw):
        self.close()
        if self.params['filename'] == '_autoplay.mp4':
            self.show()

    def show(self, **kw):
        self.close()
        fn = self.params['filename']
        display(mvp.ipython_display(fn, **kw))

!nvidia-smi -L

In [ ]:
import torch
import torchvision.models as models

torch.set_default_tensor_type('torch.cuda.FloatTensor')

In [ ]:
#@title Training Parameters

# Grid and channel settings
CHANNEL_N = 12        # Number of CA state channels (RGB + 9 hidden)
TARGET_SIZE = 40      # Target image size (same as Growing NCA)
TARGET_PADDING = 16   # Padding around target image

# Training settings (matching Growing NCA)
BATCH_SIZE = 8        # Batch size for training
POOL_SIZE = 1024      # Number of patterns in pool (same as Growing NCA)
CELL_FIRE_RATE = 0.5  # Stochastic update rate (same as Growing NCA)
DAMAGE_N = 3          # Number of patterns to damage per batch (0 = no damage)

print(f"Training configuration:")
print(f"  - Channels: {CHANNEL_N}")
print(f"  - Target size: {TARGET_SIZE} (padded to {TARGET_SIZE + 2*TARGET_PADDING})")
print(f"  - Pool size: {POOL_SIZE}")
print(f"  - Batch size: {BATCH_SIZE}")
print(f"  - Cell fire rate: {CELL_FIRE_RATE}")
print(f"  - Damage patterns per batch: {DAMAGE_N}")

In [ ]:
#@title Load Target Image {vertical-output: true}
from google.colab import files

print("Please upload your target image:")
uploaded = files.upload()

filename = list(uploaded.keys())[0]
print(f'Using uploaded file: "{filename}"')

# Load and resize target image
target_img = imread(io.BytesIO(uploaded[filename]), max_size=TARGET_SIZE, mode='RGB')
print(f"Target image shape: {target_img.shape}")

# Pad target image with zeros (black border, like Growing NCA)
p = TARGET_PADDING
target_img_padded = np.pad(target_img, [(p, p), (p, p), (0, 0)], mode='constant', constant_values=0.0)
print(f"Padded target shape: {target_img_padded.shape}")

# Convert to torch tensor [1, C, H, W]
target_torch = torch.tensor(target_img_padded).permute(2, 0, 1).unsqueeze(0)
print(f"Target tensor shape: {target_torch.shape}")

# Get grid dimensions from padded target
H, W = target_img_padded.shape[:2]
print(f"Grid size: {H} x {W}")

imshow(target_img_padded)

In [ ]:
#@title CA Model with Stochastic Updates

def depthwise_conv(x, filters):
    """filters: [filter_n, h, w]"""
    b, ch, h, w = x.shape
    y = x.reshape(b * ch, 1, h, w)
    y = torch.nn.functional.pad(y, [1, 1, 1, 1], "circular")  # Circular padding (toroidal)
    y = torch.nn.functional.conv2d(y, filters[:, None])
    return y.reshape(b, -1, h, w)

def merge_lap(z):
    # Merge lap_x and lap_y into a single laplacian filter
    b, c, h, w = z.shape  # [b, 5 * chn, h, w]
    z = torch.stack([
        z[:, ::5],
        z[:, 1::5],
        z[:, 2::5],
        z[:, 3::5] + z[:, 4::5]
    ], dim=2)  # [b, chn, 4, h, w]
    return z.reshape(b, -1, h, w)  # [b, 4 * chn, h, w]


class GrowingNCA(torch.nn.Module):
    """NCA with stochastic cell updates like Growing NCA."""
    
    def __init__(self, chn=CHANNEL_N, fc_dim=96, fire_rate=CELL_FIRE_RATE):
        super().__init__()
        self.chn = chn
        self.fire_rate = fire_rate
        
        # Same architecture as NoiseNCA
        self.w1 = torch.nn.Conv2d(chn * 4, fc_dim, 1, bias=True)
        self.w2 = torch.nn.Conv2d(fc_dim, chn, 1, bias=False)

        torch.nn.init.xavier_normal_(self.w1.weight, gain=0.2)
        torch.nn.init.zeros_(self.w2.weight)

        with torch.no_grad():
            ident = torch.tensor([[0.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 0.0]])
            sobel_x = torch.tensor([[-1.0, 0.0, 1.0], [-2.0, 0.0, 2.0], [-1.0, 0.0, 1.0]]) / 8.0  # Normalized like Growing NCA
            lap_x = torch.tensor([[1.0, 2.0, 1.0], [2.0, -12.0, 2.0], [1.0, 2.0, 1.0]])
            self.filters = torch.stack([ident, sobel_x, sobel_x.T, lap_x, lap_x.T])

    def perception(self, s):
        z = depthwise_conv(s, self.filters)  # [b, 5 * chn, h, w]
        return merge_lap(z)

    def forward(self, s, fire_rate=None, step_size=1.0):
        # Perception
        z = self.perception(s)
        
        # Neural network update
        delta_s = self.w2(torch.relu(self.w1(z))) * step_size
        
        # Stochastic cell updates (key feature from Growing NCA)
        if fire_rate is None:
            fire_rate = self.fire_rate
        
        # Create update mask: only fire_rate fraction of cells update
        update_mask = (torch.rand(s.shape[0], 1, s.shape[2], s.shape[3], device=s.device) <= fire_rate).float()
        
        return s + delta_s * update_mask

    def make_seed(self, n, h, w):
        """Create a fixed seed: zeros everywhere except center has activation in hidden channels.
        
        Same as Growing NCA but without alpha channel.
        """
        x = torch.zeros(n, self.chn, h, w)
        # Activate hidden channels at center (channels 3+)
        x[:, 3:, h//2, w//2] = 1.0
        return x


def to_rgb(s):
    """Extract RGB channels. State RGB is in [0, 1] range directly."""
    return s[..., :3, :, :].clamp(0, 1)


param_n = sum(p.numel() for p in GrowingNCA().parameters())
print(f'GrowingNCA param count: {param_n}')

In [ ]:
#@title Damage and Pool Utilities

def make_circle_masks(n, h, w):
    """Generate random circular damage masks (same as Growing NCA).
    
    Returns masks where 0.0 = damage (erase), 1.0 = keep.
    """
    # Create coordinate grids
    x = torch.linspace(-1.0, 1.0, w)[None, None, :]  # [1, 1, w]
    y = torch.linspace(-1.0, 1.0, h)[None, :, None]  # [1, h, 1]
    
    # Random centers and radii (same ranges as Growing NCA)
    center = torch.rand(2, n, 1, 1) * 1.0 - 0.5  # [-0.5, 0.5]
    r = torch.rand(n, 1, 1) * 0.3 + 0.1  # [0.1, 0.4]
    
    # Compute distance from center, normalized by radius
    x_dist = (x - center[0]) / r
    y_dist = (y - center[1]) / r
    
    # Mask: 1.0 inside circle (damage area), 0.0 outside
    # We return 1.0 - mask so that multiplying erases the circle
    inside_circle = (x_dist ** 2 + y_dist ** 2) < 1.0
    damage_mask = 1.0 - inside_circle.float()  # [n, h, w]
    return damage_mask


def compute_per_sample_loss(s, target):
    """Compute pixel-wise MSE loss for each sample in batch.
    
    Args:
        s: State tensor [b, chn, h, w]
        target: Target RGB tensor [1, 3, h, w]
    
    Returns:
        Per-sample loss tensor [b]
    """
    rgb = to_rgb(s)  # [b, 3, h, w]
    # MSE per sample (averaged over spatial and channel dims)
    return ((rgb - target) ** 2).mean(dim=(1, 2, 3))  # [b]


# Test damage masks
print("Testing damage mask generation...")
test_masks = make_circle_masks(4, H, W)
print(f"Mask shape: {test_masks.shape}")
print(f"Mask range: [{test_masks.min():.1f}, {test_masks.max():.1f}]")
imshow(np.hstack(test_masks.cpu().numpy()))

In [ ]:
#@title Setup Training
import os
from google.colab import files

# Check for checkpoint files in Colab storage
checkpoint_files = glob.glob('*.pt')

if checkpoint_files:
    print(f"Found {len(checkpoint_files)} checkpoint file(s) in Colab storage:")
    for i, f in enumerate(checkpoint_files):
        file_size_mb = os.path.getsize(f) / (1024 * 1024)
        print(f"  [{i}] {f} ({file_size_mb:.2f} MB)")

    choice = input("\nEnter the number to load a checkpoint, 'u' to upload a new file, or press Enter to start fresh: ").strip()

    if choice == 'u':
        print("Please upload your checkpoint file:")
        uploaded = files.upload()
        filename = list(uploaded.keys())[0]
        print(f'Loading checkpoint from uploaded file: "{filename}"')
        checkpoint = torch.load(filename)
    elif choice.isdigit() and 0 <= int(choice) < len(checkpoint_files):
        filename = checkpoint_files[int(choice)]
        print(f'Loading checkpoint from: "{filename}"')
        checkpoint = torch.load(filename)
    else:
        checkpoint = None
else:
    print("No checkpoint files found in Colab storage.")
    upload_choice = input("Upload a checkpoint file? (y/n, default=n): ").strip().lower()

    if upload_choice == 'y':
        print("Please upload your checkpoint file:")
        uploaded = files.upload()
        filename = list(uploaded.keys())[0]
        print(f'Loading checkpoint from uploaded file: "{filename}"')
        checkpoint = torch.load(filename)
    else:
        checkpoint = None

# Initialize or load model
if checkpoint is not None:
    model = GrowingNCA(chn=CHANNEL_N, fire_rate=CELL_FIRE_RATE)
    model.load_state_dict(checkpoint['model_state_dict'])
    start_iteration = checkpoint.get('iteration', 0) + 1
    print(f'Loaded checkpoint from iteration {start_iteration - 1}')
else:
    print('Initializing new GrowingNCA model...')
    model = GrowingNCA(chn=CHANNEL_N, fire_rate=CELL_FIRE_RATE)
    start_iteration = 0

# Optimizer with same LR as Growing NCA
lr = 2e-3
opt = torch.optim.Adam(model.parameters(), lr)

# Learning rate scheduler (piecewise constant like Growing NCA: lr -> lr*0.1 at step 2000)
lr_sched = torch.optim.lr_scheduler.MultiStepLR(opt, milestones=[2000], gamma=0.1)

# Load optimizer and scheduler state if resuming
if checkpoint is not None:
    if 'optimizer_state_dict' in checkpoint:
        opt.load_state_dict(checkpoint['optimizer_state_dict'])
    if 'scheduler_state_dict' in checkpoint:
        lr_sched.load_state_dict(checkpoint['scheduler_state_dict'])
    if 'loss_log' in checkpoint:
        loss_log = checkpoint['loss_log']
    else:
        loss_log = []
    if 'pool' in checkpoint:
        pool = checkpoint['pool']
    else:
        with torch.no_grad():
            pool = model.make_seed(POOL_SIZE, H, W)
else:
    loss_log = []
    # Initialize pool with seeds (all start from same fixed seed)
    with torch.no_grad():
        pool = model.make_seed(POOL_SIZE, H, W)

# Create the fixed seed for re-injection
with torch.no_grad():
    seed = model.make_seed(1, H, W)[0]  # [chn, h, w] single seed

print(f'Starting from iteration {start_iteration}')
print(f'Pool shape: {pool.shape}')

# Visualize initial seed
print("\nInitial seed state (RGB):")
seed_rgb = to_rgb(pool[:1]).permute(0, 2, 3, 1).cpu().numpy()[0]
imshow(seed_rgb)

In [ ]:
# @title Training Loop {vertical-output: true}
# Growing NCA-style training with:
# - Fixed seed initialization
# - Random pool sampling, then loss-ranked sorting
# - Damage augmentation
# - Pixel-wise MSE loss
# - Stochastic cell updates

# Training parameters
num_iterations = 8000  #@param {type: "integer"}

try:
    for i in range(start_iteration, start_iteration + num_iterations):
        # Sample batch randomly from pool (like Growing NCA's SamplePool.sample)
        batch_idx = np.random.choice(POOL_SIZE, BATCH_SIZE, replace=False)
        x0 = pool[batch_idx].clone()
        
        # Compute loss and sort by loss (highest first)
        with torch.no_grad():
            batch_losses = compute_per_sample_loss(x0, target_torch)
            loss_rank = batch_losses.argsort(descending=True).cpu().numpy()
            x0 = x0[loss_rank]
            batch_idx = batch_idx[loss_rank]  # Reorder indices to match
        
        # Replace first sample with fresh seed
        x0[0] = seed.clone()
        
        # Apply damage to last DAMAGE_N samples
        if DAMAGE_N > 0:
            damage_masks = make_circle_masks(DAMAGE_N, H, W)  # [DAMAGE_N, h, w]
            damage_masks = damage_masks[:, None, :, :]  # [DAMAGE_N, 1, h, w]
            x0[-DAMAGE_N:] = x0[-DAMAGE_N:] * damage_masks
        
        # Run CA for random number of steps (64-96 like Growing NCA)
        x = x0
        step_n = np.random.randint(64, 96)
        for k in range(step_n):
            x = model(x)
        
        # Compute pixel-wise MSE loss
        rgb = to_rgb(x)  # [b, 3, h, w]
        loss = ((rgb - target_torch) ** 2).mean()
        
        # Backward pass with gradient normalization (like Growing NCA)
        loss.backward()
        with torch.no_grad():
            for p in model.parameters():
                if p.grad is not None:
                    p.grad /= (p.grad.norm() + 1e-8)
        opt.step()
        opt.zero_grad()
        lr_sched.step()
        
        # Update pool with trained samples
        with torch.no_grad():
            pool[batch_idx] = x.detach()
        
        loss_log.append(loss.item())
        
        # Display progress
        if i % 10 == 0:
            current_lr = opt.param_groups[0]['lr']
            display(Markdown(f'''
        iteration: {i}
        loss: {loss.item():.4f}
        log10(loss): {np.log10(loss.item()):.3f}
        lr: {current_lr:.2e}'''), display_id='stats')
        
        # Plot and visualize
        if i % 100 == 0:
            pl.figure(figsize=(10, 4))
            pl.title('Loss history (log10)')
            pl.plot(np.log10(loss_log), '.', alpha=0.1)
            pl.xlabel('Iteration')
            pl.ylabel('log10(loss)')
            pl.tight_layout()
            imshow(grab_plot(), id='log')
            
            # Show batch results (before and after)
            with torch.no_grad():
                vis0 = to_rgb(x0).permute(0, 2, 3, 1).cpu().numpy()
                vis1 = to_rgb(x).permute(0, 2, 3, 1).cpu().numpy()
            print("Batch (before / after):")
            imshow(np.vstack([np.hstack(vis0[:4]), np.hstack(vis1[:4])]), id='batch')
        
        # Save checkpoint periodically
        if i % 2000 == 0 and i > start_iteration:
            checkpoint = {
                'iteration': i,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': opt.state_dict(),
                'scheduler_state_dict': lr_sched.state_dict(),
                'loss_log': loss_log,
                'pool': pool,
            }
            torch.save(checkpoint, f'growing_checkpoint_iter_{i}.pt')
            print(f'\nSaved checkpoint at iteration {i}')

except KeyboardInterrupt:
    print('\n\nTraining interrupted by user!')
    print(f'Saving checkpoint at iteration {i}...')
    checkpoint = {
        'iteration': i,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': opt.state_dict(),
        'scheduler_state_dict': lr_sched.state_dict(),
        'loss_log': loss_log,
        'pool': pool,
    }
    torch.save(checkpoint, f'growing_checkpoint_interrupted_iter_{i}.pt')
    print(f'Checkpoint saved as growing_checkpoint_interrupted_iter_{i}.pt')

print(f'\nTraining completed at iteration {i}')

In [ ]:
#@title Visualize Growth from Seed {vertical-output: true}

print("Growing from seed...")
with torch.no_grad():
    x = model.make_seed(1, H, W)
    frames = []
    
    for step in range(200):
        if step % 4 == 0:
            rgb = to_rgb(x)[0].permute(1, 2, 0).cpu().numpy()
            frames.append(rgb)
        x = model(x)
    
    # Final result
    rgb = to_rgb(x)[0].permute(1, 2, 0).cpu().numpy()
    frames.append(rgb)

# Show a few key frames
key_frames = [frames[0], frames[len(frames)//4], frames[len(frames)//2], frames[-1]]
print("Growth progression (seed -> 1/4 -> 1/2 -> final):")
imshow(np.hstack(key_frames))

print("\nTarget image for comparison:")
imshow(target_img_padded)

In [ ]:
#@title Test Regeneration {vertical-output: true}

print("Testing regeneration after damage...")
with torch.no_grad():
    # Grow to stable state first
    x = model.make_seed(1, H, W)
    for _ in range(150):
        x = model(x)
    
    grown = to_rgb(x)[0].permute(1, 2, 0).cpu().numpy()
    
    # Apply damage
    damage_mask = make_circle_masks(1, H, W)[:, None, :, :]  # [1, 1, h, w]
    x_damaged = x * damage_mask
    damaged = to_rgb(x_damaged)[0].permute(1, 2, 0).cpu().numpy()
    
    # Let it regenerate
    frames_regen = [damaged]
    for step in range(100):
        x_damaged = model(x_damaged)
        if step % 10 == 0:
            rgb = to_rgb(x_damaged)[0].permute(1, 2, 0).cpu().numpy()
            frames_regen.append(rgb)
    
    regenerated = to_rgb(x_damaged)[0].permute(1, 2, 0).cpu().numpy()

print("Grown -> Damaged -> Regenerated:")
imshow(np.hstack([grown, damaged, regenerated]))

print("\nRegeneration sequence:")
imshow(np.hstack(frames_regen[:6]))

In [ ]:
#@title Create Growth Video

print("Creating growth video...")
with torch.no_grad():
    x = model.make_seed(1, H, W)
    
    with VideoWriter('growth.mp4', fps=30.0) as vid:
        for step in range(300):
            rgb = to_rgb(x)[0].permute(1, 2, 0).cpu().numpy()
            vid.add(zoom(rgb, 4))
            x = model(x)
        
        # Hold final frame
        rgb = to_rgb(x)[0].permute(1, 2, 0).cpu().numpy()
        for _ in range(30):
            vid.add(zoom(rgb, 4))

print("Video saved as growth.mp4")

In [ ]:
#@title Save Model Weights
import torch
from google.colab import files

# Save weights-only file (for demo)
weights_filename = 'growing_weights.pt'
torch.save(model.state_dict(), weights_filename)
print(f"Model weights saved as '{weights_filename}'")

# Save full checkpoint (for resuming training)
checkpoint_filename = 'growing_checkpoint_final.pt'
checkpoint = {
    'iteration': i,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': opt.state_dict(),
    'scheduler_state_dict': lr_sched.state_dict(),
    'loss_log': loss_log,
    'pool': pool,
}
torch.save(checkpoint, checkpoint_filename)
print(f"Full checkpoint saved as '{checkpoint_filename}'")

# Offer to download
print("\nDownload files:")
files.download(weights_filename)